# ASTRAM AI - Model Performance Visualizations

**Flipkart Grid 2.0, Round 2**

This notebook generates publication-quality visualizations for the presentation and technical documentation.

## Outputs:
- Confusion Matrix
- Feature Importance Charts
- Prediction vs Actual Scatter Plots
- Residual Distribution Analysis
- Model Comparison Charts

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from catboost import CatBoostRegressor
from sklearn.metrics import confusion_matrix, mean_absolute_error, r2_score, mean_squared_error
from sklearn.model_selection import train_test_split
import warnings
warnings.filterwarnings('ignore')

# Set style
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

print("Libraries loaded successfully")

In [ ]:
# Load data
df = pd.read_parquet('../../astram/data/model_ready_v2.parquet')
print(f"Loaded {len(df):,} incidents")
print(f"Features: {df.shape[1]}")
df.head()

In [ ]:
# Load trained models
incident_model = CatBoostRegressor()
incident_model.load_model('../../astram/models/catboost_best.cbm')

forecast_model = CatBoostRegressor()
forecast_model.load_model('../../astram/models/forecast_event_impact.cbm')

print("Models loaded successfully")

In [ ]:
# Prepare test data (same as training split)
feature_cols = [col for col in df.columns if col not in ['impact_score', 'risk_class', 'incident_id']]
X = df[feature_cols]
y = df['impact_score']
y_class = df['risk_class']

X_train, X_test, y_train, y_test, y_class_train, y_class_test = train_test_split(
    X, y, y_class, test_size=0.2, random_state=42
)

# Get predictions
y_pred = incident_model.predict(X_test)

# Convert to risk classes
def score_to_class(score):
    if score >= 80: return 'Critical'
    if score >= 60: return 'High'
    if score >= 40: return 'Medium'
    return 'Low'

y_pred_class = [score_to_class(s) for s in y_pred]

print(f"Test set: {len(X_test):,} samples")
print(f"R² Score: {r2_score(y_test, y_pred):.4f}")
print(f"MAE: {mean_absolute_error(y_test, y_pred):.3f}")
print(f"RMSE: {np.sqrt(mean_squared_error(y_test, y_pred)):.3f}")

## 1. Confusion Matrix - Risk Classification

In [ ]:
# Confusion matrix
labels = ['Low', 'Medium', 'High', 'Critical']
cm = confusion_matrix(y_class_test, y_pred_class, labels=labels)

plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=labels, yticklabels=labels,
            cbar_kws={'label': 'Count'})
plt.title('ASTRAM AI - Incident Impact Classification\nConfusion Matrix (Test Set)', fontsize=14, weight='bold')
plt.ylabel('Actual Risk Class', fontsize=12)
plt.xlabel('Predicted Risk Class', fontsize=12)
plt.tight_layout()
plt.savefig('../docs/confusion_matrix.png', dpi=300, bbox_inches='tight')
plt.show()

print("Confusion matrix saved to ../docs/confusion_matrix.png")

## 2. Feature Importance - Top 15 Features

In [ ]:
# Get feature importance
feature_importance = incident_model.get_feature_importance()
feature_names = incident_model.feature_names_

# Create DataFrame
fi_df = pd.DataFrame({
    'feature': feature_names,
    'importance': feature_importance
}).sort_values('importance', ascending=False).head(15)

# Plot
plt.figure(figsize=(10, 8))
plt.barh(fi_df['feature'], fi_df['importance'], color='#ef4444')
plt.xlabel('Importance Score', fontsize=12)
plt.title('ASTRAM AI - Feature Importance\nTop 15 Features Contributing to Impact Prediction', 
          fontsize=14, weight='bold')
plt.gca().invert_yaxis()
plt.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.savefig('../docs/feature_importance.png', dpi=300, bbox_inches='tight')
plt.show()

print("Feature importance saved to ../docs/feature_importance.png")
print("\nTop 5 Features:")
for idx, row in fi_df.head().iterrows():
    print(f"  {row['feature']}: {row['importance']:.2f}")

## 3. Prediction vs Actual - Scatter Plot

In [ ]:
# Scatter plot
plt.figure(figsize=(10, 10))
plt.scatter(y_test, y_pred, alpha=0.5, s=20, c='#3b82f6', edgecolors='none')
plt.plot([0, 100], [0, 100], 'r--', lw=2, label='Perfect Prediction')
plt.xlabel('Actual Impact Score', fontsize=12)
plt.ylabel('Predicted Impact Score', fontsize=12)
plt.title(f'ASTRAM AI - Model Accuracy\nR² = {r2_score(y_test, y_pred):.4f}, MAE = {mean_absolute_error(y_test, y_pred):.2f}',
          fontsize=14, weight='bold')
plt.legend(fontsize=10)
plt.grid(alpha=0.3)
plt.xlim(0, 100)
plt.ylim(0, 100)
plt.tight_layout()
plt.savefig('../docs/prediction_scatter.png', dpi=300, bbox_inches='tight')
plt.show()

print("Scatter plot saved to ../docs/prediction_scatter.png")

## 4. Residual Distribution - Error Analysis

In [ ]:
# Calculate residuals
residuals = y_test - y_pred

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram
axes[0].hist(residuals, bins=50, edgecolor='black', alpha=0.7, color='#3b82f6')
axes[0].axvline(0, color='red', linestyle='--', linewidth=2, label='Zero Error')
axes[0].set_xlabel('Residual (Actual - Predicted)', fontsize=11)
axes[0].set_ylabel('Frequency', fontsize=11)
axes[0].set_title('Residual Distribution', fontsize=12, weight='bold')
axes[0].legend()
axes[0].grid(alpha=0.3)
axes[0].text(0.05, 0.95, f'Mean: {residuals.mean():.2f}\nStd: {residuals.std():.2f}',
             transform=axes[0].transAxes, verticalalignment='top',
             bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

# Q-Q Plot
from scipy import stats
stats.probplot(residuals, dist="norm", plot=axes[1])
axes[1].set_title('Q-Q Plot (Normality Check)', fontsize=12, weight='bold')
axes[1].grid(alpha=0.3)

plt.suptitle('ASTRAM AI - Residual Analysis', fontsize=14, weight='bold', y=1.02)
plt.tight_layout()
plt.savefig('../docs/residuals.png', dpi=300, bbox_inches='tight')
plt.show()

print("Residual plots saved to ../docs/residuals.png")
print(f"Residual mean: {residuals.mean():.3f} (close to 0 = unbiased)")
print(f"Residual std dev: {residuals.std():.3f}")

## 5. Risk Class Distribution

In [ ]:
# Risk class distribution
actual_counts = y_class_test.value_counts().reindex(labels, fill_value=0)
pred_counts = pd.Series(y_pred_class).value_counts().reindex(labels, fill_value=0)

x = np.arange(len(labels))
width = 0.35

plt.figure(figsize=(10, 6))
plt.bar(x - width/2, actual_counts, width, label='Actual', color='#3b82f6', alpha=0.8)
plt.bar(x + width/2, pred_counts, width, label='Predicted', color='#ef4444', alpha=0.8)
plt.xlabel('Risk Class', fontsize=12)
plt.ylabel('Count', fontsize=12)
plt.title('ASTRAM AI - Risk Class Distribution\nActual vs Predicted (Test Set)', 
          fontsize=14, weight='bold')
plt.xticks(x, labels)
plt.legend()
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('../docs/risk_distribution.png', dpi=300, bbox_inches='tight')
plt.show()

print("Risk distribution saved to ../docs/risk_distribution.png")

## 6. Model Performance Comparison

In [ ]:
# Train vs Test performance comparison
y_train_pred = incident_model.predict(X_train)

metrics = {
    'Metric': ['R²', 'MAE', 'RMSE'],
    'Train': [
        r2_score(y_train, y_train_pred),
        mean_absolute_error(y_train, y_train_pred),
        np.sqrt(mean_squared_error(y_train, y_train_pred))
    ],
    'Test': [
        r2_score(y_test, y_pred),
        mean_absolute_error(y_test, y_pred),
        np.sqrt(mean_squared_error(y_test, y_pred))
    ]
}

metrics_df = pd.DataFrame(metrics)
metrics_df['Delta'] = metrics_df['Test'] - metrics_df['Train']
metrics_df['Delta %'] = (metrics_df['Delta'] / metrics_df['Train'] * 100).round(2)

print("\n=== Model Performance: Train vs Test ===")
print(metrics_df.to_string(index=False))
print("\nConclusion: Minimal train-test gap indicates good generalization (no overfitting)")

# Visualize
fig, ax = plt.subplots(figsize=(10, 6))
x = np.arange(len(metrics_df))
width = 0.35

ax.bar(x - width/2, metrics_df['Train'], width, label='Train', color='#10b981', alpha=0.8)
ax.bar(x + width/2, metrics_df['Test'], width, label='Test', color='#3b82f6', alpha=0.8)
ax.set_xlabel('Metric', fontsize=12)
ax.set_ylabel('Score', fontsize=12)
ax.set_title('ASTRAM AI - Train vs Test Performance\nProving No Overfitting', 
             fontsize=14, weight='bold')
ax.set_xticks(x)
ax.set_xticklabels(metrics_df['Metric'])
ax.legend()
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('../docs/train_test_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

print("\nTrain-test comparison saved to ../docs/train_test_comparison.png")

## 7. Export Summary Report

In [ ]:
# Create summary report
summary = f"""
ASTRAM AI - Model Performance Summary
Flipkart Grid 2.0, Round 2
=====================================

INCIDENT IMPACT MODEL
---------------------
Dataset Size: {len(df):,} incidents
Features: {len(feature_cols)} engineered features
Train Size: {len(X_train):,} samples (80%)
Test Size: {len(X_test):,} samples (20%)

Performance Metrics:
  R² Score:  {r2_score(y_test, y_pred):.4f}
  MAE:       {mean_absolute_error(y_test, y_pred):.3f}
  RMSE:      {np.sqrt(mean_squared_error(y_test, y_pred)):.3f}

Train vs Test Delta:
  R² Delta:   {metrics_df.loc[0, 'Delta']:.4f} ({metrics_df.loc[0, 'Delta %']:.2f}%)
  MAE Delta:  +{metrics_df.loc[1, 'Delta']:.3f} ({metrics_df.loc[1, 'Delta %']:.2f}%)
  RMSE Delta: +{metrics_df.loc[2, 'Delta']:.3f} ({metrics_df.loc[2, 'Delta %']:.2f}%)

Conclusion: Minimal performance drop from train to test proves model
            generalization and absence of overfitting.

Top 5 Features:
{chr(10).join(f"  {i+1}. {row['feature']}: {row['importance']:.2f}" for i, (idx, row) in enumerate(fi_df.head().iterrows()))}

Classification Accuracy by Risk Class:
"""

# Calculate per-class accuracy
for label in labels:
    mask = y_class_test == label
    if mask.sum() > 0:
        class_acc = (np.array(y_pred_class)[mask] == label).sum() / mask.sum() * 100
        summary += f"  {label:8s}: {class_acc:.1f}% ({mask.sum()} samples)\n"

summary += f"""
Visualizations Generated:
  - confusion_matrix.png
  - feature_importance.png
  - prediction_scatter.png
  - residuals.png
  - risk_distribution.png
  - train_test_comparison.png

All charts saved to: project/docs/
"""

print(summary)

# Save to file
with open('../docs/model_performance_summary.txt', 'w') as f:
    f.write(summary)

print("\nSummary saved to ../docs/model_performance_summary.txt")

## All visualizations generated successfully!

**Files created in `project/docs/`:**
1. `confusion_matrix.png` - Classification accuracy matrix
2. `feature_importance.png` - Top 15 contributing features
3. `prediction_scatter.png` - Predicted vs Actual scatter plot
4. `residuals.png` - Error distribution analysis
5. `risk_distribution.png` - Actual vs Predicted risk class counts
6. `train_test_comparison.png` - Overfitting defense chart
7. `model_performance_summary.txt` - Text summary

**Ready for PPT and Technical Documentation**